
<table style="border: none; width: 100%; border-collapse: collapse;">
  <tr>
    <td style="border: none; text-align: center;">
      <img src="https://raw.githubusercontent.com/cristiandarioortegayubro/databricks-finance-lab/main/Imagenes/uda.jpg" width="90"/>
    </td>
    <td style="border: none">
      <h1>Universidad del Aconcagua</h1>
      <h2>Facultad de Ciencias Económicas y Jurídicas</h2>
    </td>
  </tr>
</table>

---

# Databricks Finance Lab
## Analítica Financiera Agéntica

### Módulo 02: Valoración de Instrumentos Financieros
### Notebook 2.1: Valoración de Bonos
### 📋 **PRECIO, YTM Y DURACIÓN**

---

# 2.1 - Valoración de Bonos

## Objetivo del Módulo
Comprender cómo valorar bonos y entender la relación entre precio, tasa de interés y riesgo.

## Conceptos Clave
* **Bono**: Instrumento de deuda que paga intereses periódicos (cupones) y devuelve el principal al vencimiento
* **Valor Nominal (VN)**: Monto principal del bono
* **Cupón**: Pago periódico de intereses
* **Tasa Cupón**: Tasa de interés anual del bono
* **Yield to Maturity (YTM)**: Tasa de retorno si se mantiene hasta el vencimiento
* **Precio del Bono**: Valor presente de todos los flujos futuros
* **Duración**: Medida de sensibilidad del precio a cambios en tasas
* **Convexidad**: Curvatura de la relación precio-tasa

## 📚 Referencias del Libro de Texto

**Libro**: Finanzas Corporativas - Un Enfoque Latinoamericano (2ª ed.)
**Autor**: Guillermo L. Dumrauf
**Capítulo**: 6 - Valoración de títulos valores
**Sección**: 6.1-6.3 - Valoración de bonos (pág. 147-165)
**Ubicación**: `/Workspace/Users/cortega@uda.edu.ar/Databricks Finance Lab/Libros/Finanzas corporativas.pdf`

### 📝 Contenido cubierto:

* **Pág. 147-152**: Características de los bonos y flujos de caja
* **Pág. 153-158**: Valoración de bonos con cupones
* **Pág. 159-162**: Relación precio-tasa de interés
* **Pág. 162-165**: Duración y convexidad

---

In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

UDA_COLORS = {'primary': '#003366', 'accent': '#FF6600', 'success': '#28A745', 'danger': '#DC3545'}

print("✓ Librerías cargadas correctamente")

## 📐 Fórmulas Fundamentales

### Precio de un Bono con Cupones

El precio de un bono es el **valor presente** de todos sus flujos futuros:

$$P = \sum_{t=1}^{n} \frac{C}{(1 + r)^t} + \frac{VN}{(1 + r)^n}$$

Donde:
* $P$ = Precio del bono
* $C$ = Cupón periódico (VN × tasa_cupón)
* $VN$ = Valor Nominal
* $r$ = Yield to Maturity (YTM)
* $n$ = Número de períodos hasta el vencimiento

### Bono Cupón Cero

Para bonos sin cupones:

$$P = \frac{VN}{(1 + r)^n}$$

### Relación Precio-Tasa (INVERSA)

* **YTM < Tasa Cupón** → Precio > VN (bono a **premium**)
* **YTM = Tasa Cupón** → Precio = VN (bono a **par**)
* **YTM > Tasa Cupón** → Precio < VN (bono a **discount**)

### Duración de Macaulay

$$D = \frac{\sum_{t=1}^{n} t \times \frac{C}{(1+r)^t} + n \times \frac{VN}{(1+r)^n}}{P}$$

La duración mide la **sensibilidad del precio** a cambios en las tasas de interés.

---

In [0]:
def precio_bono_cupones(vn, tasa_cupon, ytm, periodos):
    """
    Calcula el precio de un bono con cupones
    
    Parámetros:
    vn: Valor nominal del bono
    tasa_cupon: Tasa de cupón anual (decimal)
    ytm: Yield to Maturity (decimal)
    periodos: Número de períodos hasta vencimiento
    
    Retorna:
    Precio del bono
    """
    cupon = vn * tasa_cupon
    
    # Valor presente de los cupones
    vp_cupones = sum([cupon / (1 + ytm)**t for t in range(1, periodos + 1)])
    
    # Valor presente del valor nominal
    vp_principal = vn / (1 + ytm)**periodos
    
    return vp_cupones + vp_principal


def precio_bono_cero(vn, ytm, periodos):
    """
    Calcula el precio de un bono cupón cero
    
    Parámetros:
    vn: Valor nominal
    ytm: Yield to Maturity (decimal)
    periodos: Número de períodos
    
    Retorna:
    Precio del bono
    """
    return vn / (1 + ytm)**periodos


def ytm_aproximado(precio, vn, cupon_anual, periodos):
    """
    Calcula YTM aproximado usando fórmula simplificada
    
    Parámetros:
    precio: Precio actual del bono
    vn: Valor nominal
    cupon_anual: Cupón anual en pesos
    periodos: Períodos hasta vencimiento
    
    Retorna:
    YTM aproximado
    """
    numerador = cupon_anual + (vn - precio) / periodos
    denominador = (vn + precio) / 2
    return numerador / denominador


def duracion_macaulay(flujos, ytm):
    """
    Calcula la duración de Macaulay
    
    Parámetros:
    flujos: Lista de flujos de caja [cupon1, cupon2, ..., cupon_n + VN]
    ytm: Yield to Maturity (decimal)
    
    Retorna:
    Duración en períodos
    """
    vp_flujos = [flujo / (1 + ytm)**(t+1) for t, flujo in enumerate(flujos)]
    precio = sum(vp_flujos)
    
    duracion = sum([(t+1) * vp / precio for t, vp in enumerate(vp_flujos)])
    return duracion


print("✓ Funciones de valoración de bonos definidas")

## Ejemplo 1: Valoración de un Bono con Cupones

**Problema**: 
Una empresa emite un bono con las siguientes características:
* Valor Nominal: $1,000
* Tasa de cupón: 8% anual
* Vencimiento: 5 años
* Yield to Maturity (YTM) del mercado: 10%

**Pregunta**: ¿Cuál es el precio justo del bono?

---

In [0]:
# Datos del bono
vn = 1000
tasa_cupon = 0.08
ytm = 0.10
periodos = 5

# Calcular precio
precio = precio_bono_cupones(vn, tasa_cupon, ytm, periodos)

# Cupón anual
cupon = vn * tasa_cupon

print("="*60)
print("VALORACIÓN DE BONO CON CUPONES")
print("="*60)
print(f"\n📋 Características del Bono:")
print(f"  Valor Nominal: ${vn:,.2f}")
print(f"  Tasa de Cupón: {tasa_cupon*100}% anual")
print(f"  Cupón anual: ${cupon:,.2f}")
print(f"  Vencimiento: {periodos} años")
print(f"  YTM del mercado: {ytm*100}%")

print(f"\n💰 Precio del Bono: ${precio:,.2f}")

# Análisis
if precio < vn:
    print(f"\n🔻 El bono cotiza a DESCUENTO (discount)")
    print(f"   Diferencia: ${vn - precio:,.2f} ({(1 - precio/vn)*100:.2f}%)")
    print(f"   Razón: YTM ({ytm*100}%) > Tasa Cupón ({tasa_cupon*100}%)")
elif precio > vn:
    print(f"\n🔺 El bono cotiza a PREMIUM")
    print(f"   Diferencia: ${precio - vn:,.2f} ({(precio/vn - 1)*100:.2f}%)")
    print(f"   Razón: YTM ({ytm*100}%) < Tasa Cupón ({tasa_cupon*100}%)")
else:
    print(f"\n➡️ El bono cotiza a LA PAR")

print(f"\n📊 Retorno total esperado: {ytm*100}% anual")

## Ejemplo 2: Bono Cupón Cero (Zero-Coupon Bond)

**Problema**:
El gobierno emite un bono cupón cero:
* Valor Nominal: $1,000
* Vencimiento: 3 años
* YTM del mercado: 6%

**Pregunta**: ¿Cuánto debemos pagar hoy por este bono?

---

In [0]:
# Datos del bono cupón cero
vn_cero = 1000
ytm_cero = 0.06
periodos_cero = 3

# Calcular precio
precio_cero = precio_bono_cero(vn_cero, ytm_cero, periodos_cero)

print("="*60)
print("VALORACIÓN DE BONO CUPÓN CERO")
print("="*60)
print(f"\n📋 Características:")
print(f"  Valor Nominal: ${vn_cero:,.2f}")
print(f"  Cupones: NINGUNO (cupón cero)")
print(f"  Vencimiento: {periodos_cero} años")
print(f"  YTM: {ytm_cero*100}%")

print(f"\n💰 Precio del Bono: ${precio_cero:,.2f}")
print(f"\n📊 Análisis:")
print(f"  Inversión inicial: ${precio_cero:,.2f}")
print(f"  Valor al vencimiento: ${vn_cero:,.2f}")
print(f"  Ganancia total: ${vn_cero - precio_cero:,.2f}")
print(f"  Retorno total: {(vn_cero/precio_cero - 1)*100:.2f}%")
print(f"  Retorno anualizado: {ytm_cero*100}%")

# Verificación
retorno_anual = (vn_cero / precio_cero)**(1/periodos_cero) - 1
print(f"\n✓ Verificación: {retorno_anual*100:.2f}% anual")

## Ejemplo 3: Relación Inversa entre Precio y Tasa

Veamos cómo cambia el precio de nuestro bono cuando varía el YTM del mercado.

**Bono de referencia**:
* VN = $1,000
* Tasa cupón = 8%
* Vencimiento = 5 años

Graficaremos el precio para YTM entre 4% y 14%.

---

In [0]:
# Rango de YTMs a analizar
ytms = np.linspace(0.04, 0.14, 50)
precios = [precio_bono_cupones(1000, 0.08, y, 5) for y in ytms]

# Crear gráfico interactivo con Plotly
fig = go.Figure()

# Curva principal precio-tasa
fig.add_trace(go.Scatter(
    x=ytms * 100,
    y=precios,
    mode='lines',
    line=dict(color=UDA_COLORS['primary'], width=3),
    name='Precio del Bono',
    hovertemplate='YTM: %{x:.2f}%<br>Precio: $%{y:,.2f}<extra></extra>'
))

# Punto "a la par" (YTM = tasa cupón)
fig.add_trace(go.Scatter(
    x=[8],
    y=[1000],
    mode='markers',
    marker=dict(size=15, color=UDA_COLORS['danger'], symbol='circle',
               line=dict(width=2, color='black')),
    name='A la Par (YTM = Tasa Cupón)',
    hovertemplate='<b>A la Par</b><br>YTM: %{x}%<br>Precio: $%{y:,.2f}<extra></extra>'
))

# Línea de valor nominal
fig.add_hline(
    y=1000,
    line_dash='dash',
    line_color='gray',
    annotation_text='Valor Nominal = $1,000',
    annotation_position='right'
)

# Zonas de premium y discount (usando shapes)
fig.add_vrect(
    x0=4, x1=8,
    fillcolor=UDA_COLORS['success'], opacity=0.1,
    layer='below', line_width=0,
    annotation_text='PREMIUM<br>(Precio > VN)',
    annotation_position='top left'
)

fig.add_vrect(
    x0=8, x1=14,
    fillcolor=UDA_COLORS['danger'], opacity=0.1,
    layer='below', line_width=0,
    annotation_text='DISCOUNT<br>(Precio < VN)',
    annotation_position='top right'
)

# Configurar layout
fig.update_layout(
    title=dict(
        text='Relación INVERSA entre Precio y Tasa de Interés<br><sub>Bono VN=$1,000, Cupón=8%, 5 años</sub>',
        font=dict(size=18, family='Arial, sans-serif', color=UDA_COLORS['primary']),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title=dict(text='Yield to Maturity (%)', font=dict(size=14)),
        gridcolor='#E5E5E5',
        showgrid=True,
        range=[4, 14]
    ),
    yaxis=dict(
        title=dict(text='Precio del Bono ($)', font=dict(size=14)),
        gridcolor='#E5E5E5',
        showgrid=True,
        tickformat='$,.0f'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    showlegend=True,
    legend=dict(
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.02,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#E5E5E5',
        borderwidth=1
    ),
    hovermode='closest'
)

config = {
    'locale': 'es',
    'displayModeBar': True,
    'displaylogo': False,
    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'relacion_precio_tasa_bono',
        'height': 700,
        'width': 1400,
        'scale': 2
    }
}

fig.show(config=config)

print("📊 Observaciones:")
print("1. La relación es INVERSA: ↑ YTM → ↓ Precio")
print("2. La relación es NO LINEAL (curva convexa)")
print("3. Cuando YTM = Tasa Cupón → Precio = VN (a la par)")
print("4. Cuando YTM < Tasa Cupón → Precio > VN (premium)")
print("5. Cuando YTM > Tasa Cupón → Precio < VN (discount)")
print("\n💡 TIP: Pasa el mouse sobre la curva para explorar valores exactos")

## 📏 Duración de Macaulay

La **duración** es una medida del tiempo promedio ponderado hasta recibir los flujos de caja.

### ¿Para qué sirve?

* Mide la **sensibilidad del precio** a cambios en tasas
* Aproxima el cambio porcentual en precio: 
  $$\Delta P \approx -D \times \Delta r \times P$$
* Útil para gestión de riesgo de tasas (inmunización)

### Interpretación:

* **Duración alta** → Mayor sensibilidad a tasas → Mayor riesgo
* **Duración baja** → Menor sensibilidad a tasas → Menor riesgo

---

In [0]:
# Calcular duración para nuestro bono ejemplo
vn = 1000
tasa_cupon = 0.08
ytm = 0.10
periodos = 5

cupon = vn * tasa_cupon

# Crear lista de flujos de caja
flujos = [cupon] * (periodos - 1) + [cupon + vn]

# Calcular duración
duracion = duracion_macaulay(flujos, ytm)
precio_actual = precio_bono_cupones(vn, tasa_cupon, ytm, periodos)

print("="*60)
print("DURACIÓN DE MACAULAY")
print("="*60)
print(f"\n📋 Bono:")
print(f"  VN = ${vn:,.2f}")
print(f"  Cupón = {tasa_cupon*100}%")
print(f"  YTM = {ytm*100}%")
print(f"  Vencimiento = {periodos} años")
print(f"  Precio = ${precio_actual:,.2f}")

print(f"\n⏱️ Duración de Macaulay: {duracion:.2f} años")

print(f"\n📊 Interpretación:")
print(f"  Si las tasas suben 1%, el precio caerá aprox. {duracion:.2f}%")
print(f"  Si las tasas bajan 1%, el precio subirá aprox. {duracion:.2f}%")

# Verificar con cambio de 1%
ytm_nuevo = ytm + 0.01
precio_nuevo = precio_bono_cupones(vn, tasa_cupon, ytm_nuevo, periodos)
cambio_real = (precio_nuevo - precio_actual) / precio_actual * 100

print(f"\n✓ Verificación (YTM ↑ 1%):")
print(f"  Cambio predicho: -{duracion:.2f}%")
print(f"  Cambio real: {cambio_real:.2f}%")

## 🇦🇷 Bonos Argentinos: Contexto Local

### Tipos de Bonos en Argentina

1. **Bonos Soberanos** (Tesoro Nacional)
   * En pesos (ARS) o dólares (USD)
   * Ejemplos: Bonares, Globales (GD30, GD35, GD38, GD41, GD46)
   * Alto riesgo país → Altos rendimientos

2. **Bonos Provinciales**
   * Emitidos por provincias argentinas
   * Ejemplos: Mendoza, Buenos Aires, Córdoba
   * Riesgo variable según provincia

3. **Obligaciones Negociables (ONs)**
   * Emitidas por empresas privadas
   * Ejemplos: YPF, Pampa Energía, Telecom
   * Rating corporativo

### Particularidades del Mercado Argentino

* **Alta inflación** → Bonos CER (ajustados por inflación)
* **Volatilidad cambiaria** → Preferencia por bonos en USD
* **Riesgo país elevado** → Spreads sobre bonos del tesoro USA
* **Reestructuraciones frecuentes** → Descuentos (haircuts) históricos

### Ejemplo Real: Bono GD30

* Bono Global 2030 en USD
* Tasa cupón: ~1% anual
* Cotiza con alto descuento (YTM >> tasa cupón)
* Refleja riesgo de default soberano

---

In [0]:
# Ejemplo: Bono corporativo argentino
print("="*60)
print("EJEMPLO: OBLIGACIÓN NEGOCIABLE ARGENTINA")
print("="*60)

vn_on = 1000  # USD
tasa_cupon_on = 0.08  # 8% en USD (tasa alta para Argentina)
periodos_on = 3
ytm_argentina = 0.15  # 15% - refleja riesgo argentino

precio_on = precio_bono_cupones(vn_on, tasa_cupon_on, ytm_argentina, periodos_on)

print(f"\n📋 Obligación Negociable (ON) en USD:")
print(f"  Emisor: Empresa argentina de primera línea")
print(f"  Valor Nominal: USD {vn_on:,.2f}")
print(f"  Tasa Cupón: {tasa_cupon_on*100}% anual")
print(f"  Vencimiento: {periodos_on} años")
print(f"  YTM exigida (riesgo país): {ytm_argentina*100}%")

print(f"\n💰 Precio: USD {precio_on:,.2f}")
print(f"  Descuento: USD {vn_on - precio_on:,.2f} ({(1 - precio_on/vn_on)*100:.1f}%)")

print(f"\n🌎 Comparación con bono USA:")
ytm_usa = 0.04  # 4% - tasa libre de riesgo
precio_usa = precio_bono_cupones(vn_on, tasa_cupon_on, ytm_usa, periodos_on)
print(f"  Si fuera bono USA (4% YTM): USD {precio_usa:,.2f}")
print(f"  Diferencia (riesgo país): USD {precio_usa - precio_on:,.2f}")

riesgo_pais = ytm_argentina - ytm_usa
print(f"\n🚨 Riesgo país implícito: {riesgo_pais*100:.0f}% anual")
print(f"   ({riesgo_pais*10000:.0f} puntos básicos)")

## 📈 Curva de Rendimientos (Yield Curve)

La **curva de rendimientos** muestra la relación entre el vencimiento de los bonos y su YTM.

### Formas de la Curva:

1. **Normal (ascendente)**: Bonos largo plazo → Mayor YTM
   * Refleja expectativas de crecimiento económico

2. **Invertida (descendente)**: Bonos largo plazo → Menor YTM
   * Señal de recesión inminente

3. **Plana**: YTM similar para todos los vencimientos
   * Incertidumbre económica

---

In [0]:
# Simular curvas de rendimientos
vencimientos = np.array([1, 2, 3, 5, 7, 10, 15, 20, 30])

# Curva normal (economía estable)
ytm_normal = 0.03 + 0.02 * (1 - np.exp(-vencimientos/10))

# Curva invertida (recesión)
ytm_invertida = 0.05 - 0.015 * (vencimientos / 30)

# Curva argentina (alto riesgo)
ytm_argentina_curva = 0.12 + 0.03 * (1 - np.exp(-vencimientos/8))

# Crear gráfico interactivo con Plotly
fig = go.Figure()

# Curva Normal (USA)
fig.add_trace(go.Scatter(
    x=vencimientos,
    y=ytm_normal * 100,
    mode='lines+markers',
    line=dict(color=UDA_COLORS['success'], width=2.5),
    marker=dict(size=8, symbol='circle'),
    name='Curva Normal (USA)',
    hovertemplate='<b>USA - Curva Normal</b><br>Vencimiento: %{x} años<br>YTM: %{y:.2f}%<extra></extra>'
))

# Curva Invertida (Recesión)
fig.add_trace(go.Scatter(
    x=vencimientos,
    y=ytm_invertida * 100,
    mode='lines+markers',
    line=dict(color=UDA_COLORS['danger'], width=2.5),
    marker=dict(size=8, symbol='square'),
    name='Curva Invertida (Recesión)',
    hovertemplate='<b>Recesión - Curva Invertida</b><br>Vencimiento: %{x} años<br>YTM: %{y:.2f}%<extra></extra>'
))

# Curva Argentina (Alto Riesgo)
fig.add_trace(go.Scatter(
    x=vencimientos,
    y=ytm_argentina_curva * 100,
    mode='lines+markers',
    line=dict(color=UDA_COLORS['accent'], width=2.5),
    marker=dict(size=8, symbol='triangle-up'),
    name='Curva Argentina (Alto riesgo)',
    hovertemplate='<b>Argentina - Alto Riesgo</b><br>Vencimiento: %{x} años<br>YTM: %{y:.2f}%<extra></extra>'
))

# Configurar layout
fig.update_layout(
    title=dict(
        text='Curvas de Rendimientos Comparadas<br><sub>USA vs Recesión vs Argentina</sub>',
        font=dict(size=18, family='Arial, sans-serif', color=UDA_COLORS['primary']),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title=dict(text='Vencimiento (años)', font=dict(size=14)),
        gridcolor='#E5E5E5',
        showgrid=True
    ),
    yaxis=dict(
        title=dict(text='Yield to Maturity (%)', font=dict(size=14)),
        gridcolor='#E5E5E5',
        showgrid=True
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    showlegend=True,
    legend=dict(
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.02,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#E5E5E5',
        borderwidth=1
    ),
    hovermode='closest'
)

config = {
    'locale': 'es',
    'displayModeBar': True,
    'displaylogo': False,
    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'curvas_rendimientos_comparadas',
        'height': 700,
        'width': 1400,
        'scale': 2
    }
}

fig.show(config=config)

print("📊 Análisis de Curvas:")
print("\n🟢 Curva Normal (USA):")
print("   - Bonos corto plazo: 3-4%")
print("   - Bonos largo plazo: 4-5%")
print("   - Refleja: Economía estable, expectativa de crecimiento")

print("\n🔴 Curva Invertida:")
print("   - Bonos corto plazo: 5%")
print("   - Bonos largo plazo: 3.5%")
print("   - Refleja: Expectativa de recesión, políticas restrictivas")

print("\n🟠 Curva Argentina:")
print("   - Bonos corto plazo: 12-14%")
print("   - Bonos largo plazo: 15%")
print("   - Refleja: Alto riesgo país, historial de defaults")

print("\n💡 TIP: Pasa el mouse sobre las curvas para comparar YTM por vencimiento")

## 💪 Ejercicios Prácticos

### Ejercicio 1: Valoración Básica
Una empresa argentina emite un bono con:
* VN = USD 1,000
* Tasa cupón = 10% anual
* Vencimiento = 4 años
* YTM del mercado = 12%

**Tareas**:
a) Calcule el precio del bono
b) ¿Cotiza a premium, par o discount? ¿Por qué?
c) Si las tasas bajan a 9%, ¿qué pasará con el precio?

---

### Ejercicio 2: Bono Cupón Cero
El gobierno emite LETES (bonos cupón cero) a 180 días:
* VN = ARS 100,000
* Plazo = 0.5 años
* YTM = 80% anual (inflación alta en Argentina)

**Tareas**:
a) Calcule cuánto debe pagar hoy
b) Calcule su ganancia total
c) Compare con tasa de inflación esperada

---

### Ejercicio 3: Duración
Calcule la duración de Macaulay para un bono:
* VN = USD 1,000
* Cupón = 6%
* Vencimiento = 3 años
* YTM = 8%

**Tareas**:
a) Calcule la duración
b) Si las tasas suben 0.5%, estime el cambio de precio
c) Calcule el cambio real y compare

---

### Ejercicio 4: Comparación de Bonos
Compare estos dos bonos y decida cuál comprar:

**Bono A**:
* VN = USD 1,000
* Cupón = 8%
* Vencimiento = 5 años
* Precio = USD 950

**Bono B**:
* VN = USD 1,000
* Cupón = 6%
* Vencimiento = 5 años
* Precio = USD 900

**Tareas**:
a) Calcule el YTM aproximado de cada uno
b) ¿Cuál tiene mejor retorno?
c) Calcule la duración de ambos. ¿Cuál es más riesgoso?

---

In [0]:
# Resuelva los ejercicios aquí

# Ejercicio 1
print("="*60)
print("EJERCICIO 1: Valoración Básica")
print("="*60)
# Su código aquí



print("\n" + "="*60)
print("EJERCICIO 2: Bono Cupón Cero (LETES)")
print("="*60)
# Su código aquí



print("\n" + "="*60)
print("EJERCICIO 3: Duración")
print("="*60)
# Su código aquí



print("\n" + "="*60)
print("EJERCICIO 4: Comparación de Bonos")
print("="*60)
# Su código aquí

## 🤖 Consultas Sugeridas con Genie Code

Prueba estas consultas con Genie para profundizar tu análisis:

### Análisis Básico
* "Calcula el precio de un bono con VN=$1000, cupón 7%, 10 años, YTM=8%"
* "¿Cuál es el YTM de un bono que cotiza a $920, VN=$1000, cupón 6%, 5 años?"
* "Explica por qué los bonos caen cuando suben las tasas de interés"

### Análisis de Sensibilidad
* "Grafica cómo cambia el precio de este bono cuando YTM varía entre 5% y 15%"
* "Compara la sensibilidad de bonos a 5, 10 y 30 años ante cambio de tasas"
* "¿Qué bono es más sensible: cupón 3% o cupón 9%?"

### Duración y Riesgo
* "Calcula la duración modificada de este portafolio de bonos"
* "¿Cuánto cambiará el precio si las tasas suben 50 puntos básicos?"
* "Construye un portafolio de bonos con duración objetivo de 7 años"

### Contexto Argentino
* "Analiza por qué los bonos argentinos tienen YTM tan alto"
* "Compara bonos CER vs bonos en USD para protección inflacionaria"
* "Calcula el riesgo país implícito de un bono argentino"
* "¿Cómo afecta una reestructuración de deuda al valor de los bonos?"

### Análisis Avanzado
* "Construye una curva de rendimientos con datos de bonos del tesoro"
* "Calcula la convexidad de este bono"
* "Estima el precio limpio (clean price) vs precio sucio (dirty price)"
* "Analiza el impacto fiscal de intereses de bonos"

---

## 📚 Resumen y Conclusiones

### Conceptos Clave Aprendidos

1. **Valoración de Bonos**
   * El precio es el valor presente de todos los flujos futuros
   * Fórmula: VP(cupones) + VP(valor nominal)

2. **Relación Precio-Tasa**
   * **INVERSA**: ↑ tasas → ↓ precio
   * **NO LINEAL**: La curva es convexa
   * Premium/Par/Discount según YTM vs Tasa Cupón

3. **Duración**
   * Mide sensibilidad del precio a cambios en tasas
   * Útil para gestión de riesgo (inmunización)
   * Duración alta = Mayor riesgo

4. **Contexto Argentino**
   * Alto riesgo país → Altos rendimientos exigidos
   * Bonos CER para protección inflacionaria
   * Volatilidad y reestructuraciones frecuentes

### Aplicaciones Prácticas

* **Inversión**: Evaluar si un bono está bien precio
* **Gestión de Riesgo**: Medir exposición a tasas
* **Portafolios**: Diversificación con renta fija
* **Arbitraje**: Detectar oportunidades de valor

### Próximos Pasos

* **Notebook 2.2**: Valoración de Acciones (DDM, P/E, múltiplos)
* **Notebook 3.1**: Ratios Financieros para analizar emisores
* **Notebook 4.1**: Riesgo y Retorno de bonos en portafolios

### Referencias Adicionales

* **Dumrauf**: Capítulo 6, págs. 147-165
* **Comisión Nacional de Valores (CNV)**: Datos de bonos argentinos
* **BYMA**: Cotizaciones en tiempo real
* **Banco Central (BCRA)**: Estadísticas de tasas

---

## 🎯 ¡Has completado el módulo de Valoración de Bonos!

**Siguiente:** [2.2 - Valoración de Acciones](#notebook/1265694461779443)

---